<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Tips & Techniques - Temporal Example (Load Data)
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style="font-size:24px;font-family:Arial;color:#00233C"><b>Introduction</b></p>

<p style="font-size:16px;font-family:Arial;color:#00233C">
This notebook demonstrates how to load the commercial real estate data into a Teradata database using Python, to use in the Temporal Example Notebook.
</p>

---

<p style="font-size:24px;font-family:Arial;color:#00233C"><b>Notebook Workflow Steps</b></p>

<div style="font-size:16px;font-family:Arial;color:#00233C">
<ol>
<li>
<b>Set Up</b>
<ul>
<li>Establish a secure connection to Teradata Vantage using authentication credentials</li>
<li>Import neccesary libraries</li>
</ul>
</li>

<li>
<b>Table Creation</b>
<ul>
<li>Creating a table to insert the real estate data</li>
</ul>
</li>

<li>
<b>Data Loading</b>
<ul>
<li>Reading CSV data with pandas and loading it into Teradata using `teradataml`</li>
</ul>
</li>

</ol>
</div>


<p style="font-size:24px;font-family:Arial;color:#00233C"><b>Dataset Overview</b></p>

<ul style="font-size:16px;font-family:Arial;color:#00233C">
  <li>Property details (title, address, type)</li>
  <li>Pricing information</li>
  <li>Geographic coordinates (latitude/longitude)</li>
  <li>Property specifications (area, NBN availability)</li>
  <li>Detailed property descriptions</li>
</ul>

<p style="font-size:24px;font-family:Arial;color:#00233C"><b>Prerequisites</b></p>

<p style="font-size:16px;font-family:Arial;color:#00233C">
Before running this notebook, ensure you have:
</p>

<ul style="font-size:16px;font-family:Arial;color:#00233C">
  <li>Access to a Teradata Vantage system with valid credentials</li>
  <li>The `commercial_real_estate.csv` file in the same directory</li>
</ul>

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>1. Set Up </b>

#### 🗂️ Database Connection

Make sure you have valid database credentials (hostname, username, password) to connect succesfully to Teradata Vantage.

In [ ]:
# Connect to Vantage using create_context.
hostname = getpass(prompt = 'hostname: ')
username = getpass(prompt = 'username: ')
password = getpass(prompt = 'password: ')

hostname:  ········
username:  ········
password:  ········


In [ ]:
context=create_context(host=hostname, username=username, password=password)

/opt/conda/lib/python3.10/site-packages/teradatasqlalchemy/telemetry/queryband.py:382: UserWarning: [Teradata][teradataml](TDML_2002) Overwriting an existing context associated with Teradata Vantage Connection. Most of the operations on any teradataml DataFrames created before this will not work.
  return exposed_func(*args, **kwargs)
Exception closing connection TeradataConnection uConnHandle=1
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/site-packages/sqlalchemy/pool/base.py", line 379, in _close_connection
    self._dialect.do_close(connection)
  File "/opt/conda/lib/python3.10/site-packages/sqlalchemy/engine/default.py", line 702, in do_close
    dbapi_connection.close()
  File "/opt/conda/lib/python3.10/site-packages/teradatasql/__init__.py", line 214, in close
    raise OperationalError(sErr)
teradatasql.OperationalError: sql: connection is already closed


#### 💻 Import libraries

In [ ]:
# Required imports
from teradatasqlalchemy.types import INTEGER
from teradataml import create_context, set_auth_token, execute_sql, DataFrame
import pandas as pd
from getpass import getpass

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>2. Table Creation </b>

Create `commercial_real_estate_example` table to store the desired data with the necessary columns.

In [12]:
execute_sql("""
CREATE MULTISET TABLE df120645.commercial_real_estate_example (
    id INTEGER NOT NULL, -- Assuming this is the primary key
    titlename VARCHAR(500), -- Title of the property
    price VARCHAR(50), -- Price as a string (includes symbols like $)
    nbn VARCHAR(100), -- NBN service availability
    address VARCHAR(255), -- Address of the property
    text VARCHAR(64000), -- Detailed description of the property
    area VARCHAR(50), -- Area as a string (e.g., "175m²")
    propertytype VARCHAR(50), -- Type of property (e.g., Retail, Offices)
    latitude FLOAT, -- Latitude of the property
    longitude FLOAT -- Longitude of the property
) PRIMARY INDEX (id);
""")

TeradataCursor uRowsHandle=20 bClosed=False

<hr style="height:2px;border:none;background-color:#00233C;">
<b style = 'font-size:30px;font-family:Arial;color:#00233C'>3. Data Loading </b>

### 🚀 Data Preparation and Insertion

1. **Load the complete dataset from CSV**
2. **Ensure the index name and type is correct**
3. **Move the data into the table using Teradataml**

In [14]:
df = pd.read_csv('commercial_real_estate.csv', engine='python')

In [ ]:
# Ensure the index is named 'id' and is of type integer
df.index.name = 'id'
df.reset_index(inplace=True)  # Move the index into a column
df['id'] = df['id'].astype(int)  # Ensure id column is integer type

# Define data types for database insertion
types = {
    "id": INTEGER
}

In [ ]:
DataFrame.to_sql(df, table_name='commercial_real_estate_example', if_exists='replace', types=types)